# Show3D

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/docs/tutorials/show3d.ipynb)

`Show3D` scrubs a 3D stack slice by slice: a focal series, time series, tomographic reconstruction, or a sequence of related images. Drag the slider, press the play controls, or use the arrow keys to move through the stack.

This tutorial uses a real gold HAADF image from [`bobleesj/quantem-data`](https://huggingface.co/datasets/bobleesj/quantem-data). The built-in tutorial loader makes a calibrated moving-crop stack from that image, so the documentation and Colab examples use real microscope data while still loading quickly.

```{tip}
Run this exact notebook with the Colab badge above, or [View or download this notebook on GitHub](https://github.com/bobleesj/quantem.widget/blob/main/docs/tutorials/show3d.ipynb). For finished results, use [Saving and sharing](widget_export) to export interactive HTML or share a trusted notebook with widget state.
```


In [1]:
import subprocess
import sys

import numpy as np

try:
    import google.colab  # noqa: F401
except Exception:
    pass
else:
    from google.colab import output

    output.enable_custom_widget_manager()
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/bobleesj/quantem.widget.git"],
        check=True,
    )

from quantem.widget import Show3D
from quantem.widget.data import load_tutorial_show3d

volume_dataset = load_tutorial_show3d(n_frames=32, stride=8, crop_size=256)



Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Source: gold_haadf_npy from Hugging Face
Full image: 4096 x 4096 uint16
Stack: (32, 256, 256), stride 8, pixel size 0.1489 nm


## Scrub the real-data stack

The helper returns a quantem `Dataset3d`, so depth and lateral calibration travel with the stack. `Show3D` reads that metadata automatically and draws a physical scale bar without widget-level pixel-size arguments.

In [2]:
Show3D(volume_dataset, offline=True)

Show3D(32×256×256, frame=16, cmap=plasma)

## Trigger a fresh render in an existing widget

For live stacks, display one `Show3D` object and update it with `set_image()` as new frames arrive. `set_image()` is the render trigger: it writes a fresh current-frame transfer, bumps the frame sequence used by the frontend renderer, invalidates playback buffers, and clamps the current slice to the new stack.

Use `offline=False` for live-growing stacks. Small examples can otherwise choose the offline notebook path, which is useful for saved notebooks but is not the path you want when a loop keeps appending frames.


In [ ]:
live_frames = [frame for frame in volume_dataset.array[:4]]
live3d = Show3D(
    np.stack(live_frames),
    labels=[f"frame {i + 1}" for i in range(len(live_frames))],
    offline=False,
    fps=4,
)
live3d


In [ ]:
live_frames.extend(volume_dataset.array[4:8])
live3d.set_image(
    np.stack(live_frames),
    labels=[f"frame {i + 1}" for i in range(len(live_frames))],
)
live3d.slice_idx = len(live_frames) - 1
